# contact-CLASP · clasp_l5 — the benchmark, end to end

Pulls clasp_l1–l4 together: one call computes attention / DCA / fusion precision@L across Neff/L bins. This is the miniature of the real PDB-wide benchmark; scale-up = many chains, real MSAs, ESM-2 650M, bootstrap CIs, identity stratification (`CLAUDE.md`).

In [1]:
import os, sys
ROOT = os.path.abspath("")
while ROOT != os.path.dirname(ROOT) and not os.path.isdir(os.path.join(ROOT, "common")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)                          # for `import common`
sys.path.insert(0, os.path.join(ROOT, "clasp"))   # for `import clasp_common`
DATA = os.path.join(ROOT, "data")
print("repo root:", ROOT)

repo root: C:\Users\soura\code\2026\xai-starter


In [2]:
from clasp_common import run_clasp_experiment
m = run_clasp_experiment(neff_fractions=(1.0, 0.3, 0.1))
for k, v in m.items():
    print(f'{k:24s}: {v}')

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

L                       : 58
n_true_contacts         : 123
precision_attention     : 0.06896551724137931
precision_gradient      : 0.15517241379310345
neff_per_L@1            : 7.101
precision_dca@1         : 0.7931034482758621
precision_fusion@1      : 0.1206896551724138
neff_per_L@0.3          : 2.366
precision_dca@0.3       : 0.7586206896551724
precision_fusion@0.3    : 0.1206896551724138
neff_per_L@0.1          : 0.818
precision_dca@0.1       : 0.41379310344827586
precision_fusion@0.1    : 0.1206896551724138


### Honest read
- On the tiny 8M model attention precision@L is low — expected; the *axis* (DCA falling as Neff/L drops) is the point, and it reproduces here.
- A real verdict needs ESM-2 650M, real MSAs, and many chains with CIs.
### Things to experiment with
- Wrap this in an MLflow runner (mirror `pinch/run_pinch.py`) to sweep models and chains.